# 03 - Train / validation / test split

**Job:** split before detailed EDA so test information cannot influence
feature and model decisions.

**Decision:** use a chronological 70% / 15% / 15% split. A production
model is trained on past orders and predicts later ones; a time split
tests that behavior and exposes drift that a random split can hide.

**Input:** `artifacts/02_labeled_table.csv.gz`

**Output:** three split files and a split summary.

In [1]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks": ROOT = ROOT.parent
ARTIFACT_DIR = ROOT / "artifacts"
input_path = ARTIFACT_DIR / "02_labeled_table.csv.gz"
assert input_path.exists(), "Run Notebook 02 first."

labeled = pd.read_csv(input_path, low_memory=False)
labeled["order_purchase_timestamp"] = pd.to_datetime(
    labeled["order_purchase_timestamp"], errors="coerce"
)
assert labeled["order_purchase_timestamp"].notna().all()
labeled = labeled.sort_values(
    ["order_purchase_timestamp", "order_id"], kind="mergesort"
).reset_index(drop=True)

print(
    f"Available purchase dates: {labeled['order_purchase_timestamp'].min()} "
    f"to {labeled['order_purchase_timestamp'].max()}"
)
print(f"Overall late rate: {labeled['late'].mean():.3%}")

Available purchase dates: 2016-09-15 12:16:38 to 2018-08-29 15:00:37
Overall late rate: 8.112%


## Create chronological boundaries

In [2]:
n_rows = len(labeled)
train_end = int(n_rows * 0.70)
validation_end = int(n_rows * 0.85)

splits = {
    "train": labeled.iloc[:train_end].copy(),
    "validation": labeled.iloc[train_end:validation_end].copy(),
    "test": labeled.iloc[validation_end:].copy(),
}

assert sum(len(frame) for frame in splits.values()) == n_rows
assert set(splits["train"]["order_id"]).isdisjoint(splits["validation"]["order_id"])
assert set(splits["train"]["order_id"]).isdisjoint(splits["test"]["order_id"])
assert set(splits["validation"]["order_id"]).isdisjoint(splits["test"]["order_id"])
assert splits["train"]["order_purchase_timestamp"].max() <= splits["validation"]["order_purchase_timestamp"].min()
assert splits["validation"]["order_purchase_timestamp"].max() <= splits["test"]["order_purchase_timestamp"].min()

## Check date ranges and label balance

A chronological split does not force equal label ratios. Any shift is
part of the realistic future-data challenge and is reported rather
than corrected using test information.

In [3]:
summary_rows = []
for name, frame in splits.items():
    summary_rows.append({
        "split": name,
        "rows": len(frame),
        "share": len(frame) / n_rows,
        "purchase_start": frame["order_purchase_timestamp"].min(),
        "purchase_end": frame["order_purchase_timestamp"].max(),
        "late_orders": int(frame["late"].sum()),
        "late_rate": float(frame["late"].mean()),
    })
split_summary = pd.DataFrame(summary_rows)
display(split_summary)

max_shift = float(split_summary["late_rate"].max() - split_summary["late_rate"].min())
print(f"Maximum late-rate shift across time splits: {max_shift:.2%} points")

,split,rows,share,purchase_start,purchase_end,late_orders,late_rate
0,train,67529,0.700000,2016-09-15 12:16:38,2018-04-15 20:12:35,6096,0.090272
1,validation,14470,0.149995,2018-04-15 20:17:11,2018-06-21 08:29:29,773,0.053421
2,test,14471,0.150005,2018-06-21 08:41:07,2018-08-29 15:00:37,957,0.066132


Maximum late-rate shift across time splits: 3.69% points


## Save the step-3 artifacts

In [4]:
for name, frame in splits.items():
    path = ARTIFACT_DIR / f"03_{name}.csv.gz"
    frame.to_csv(path, index=False, compression="gzip")
    assert path.exists() and path.stat().st_size > 0

split_summary.to_csv(ARTIFACT_DIR / "03_split_summary.csv", index=False)
split_metadata = {
    "method": "chronological",
    "ratios": {"train": 0.70, "validation": 0.15, "test": 0.15},
    "sort_columns": ["order_purchase_timestamp", "order_id"],
    "reason": "simulate training on past orders and predicting later orders",
    "max_late_rate_shift": max_shift,
}
(ARTIFACT_DIR / "03_split_metadata.json").write_text(
    json.dumps(split_metadata, indent=2), encoding="utf-8"
)
print("Saved chronological train, validation, and test files.")

Saved chronological train, validation, and test files.
